In [15]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [89]:
import torch
import pandas as pd
import json
import re
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import GRPOConfig, GRPOTrainer
from sentence_transformers import SentenceTransformer, util

### Unsloth

In [16]:
# ## Model Loading
# We'll load the same Llama-3.2 model. We need to load it in a way that allows us to first generate `rejected` responses before applying the LoRA adapters for training.

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2025.7.1: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [92]:
tokenizer = get_chat_template(
    tokenizer, chat_template="llama-3.1"
)
print("✅ Model and tokenizer loaded.")

✅ Model and tokenizer loaded.


In [97]:

# ## Apply PEFT Adapters
# Now that we have our dataset, we can prepare the model for LoRA training.

# In[4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()


Unsloth: Already have LoRA adapters! We shall skip this step.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [91]:

## 2. PREPARE THE DATASET
# ==================================
## Make sure more than 8 rows of data as Unsloth grpo won't accept single or less than 8 data sample due its complexity.
# The `create_grpo_dataset` function formats data into the required
# 'prompt', 'chosen', 'rejected', and 'answer' columns for training and reward calculation.
def create_grpo_dataset(input_file):
    df = pd.read_csv(input_file)
    FastLanguageModel.for_inference(model)
    dataset_rows = []
    for _, row in df.iterrows():
        messages = [{"role": "system", "content": str(row.get('system', ''))},
                    {"role": "user", "content": str(row.get('user', ''))}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
        outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True, temperature=1.0)
        rejected = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        rejected = rejected[len(tokenizer.decode(inputs[0], skip_special_tokens=True)):]

        dataset_rows.append({
            "prompt": prompt,
            "chosen": str(row.get('assistant', '')),
            "rejected": rejected.strip(),
            "answer": str(row.get('assistant', ''))
        })
    return Dataset.from_list(dataset_rows)

dataset = create_grpo_dataset('/content/problems.csv')
print("✅ Dataset prepared for GRPO training.")

✅ Dataset prepared for GRPO training.


In [93]:
dataset[3]['chosen']


'[{"area": "Phishing Attack", "nature": "credential_harvesting", "description": "Attackers send emails impersonating trusted entities to trick users into entering their login credentials on fake websites, leading to account compromise.", "risk_reduction": ["Implement multi-factor authentication (MFA) across all accounts", "Deploy advanced email filtering and anti-phishing tools", "Conduct regular user training and phishing simulation exercises"]}, {"area": "Phishing Attack", "nature": "spear_phishing", "description": "Highly targeted phishing campaigns use personalized information to deceive specific individuals, often executives, to gain access to sensitive data or systems.", "risk_reduction": ["Use threat intelligence to identify and block targeted phishing attempts", "Educate high-risk users on recognizing spear phishing tactics", "Enforce strict verification protocols for sensitive requests"]}, {"area": "Phishing Attack", "nature": "business_email_compromise", "description": "Attac

In [94]:
dataset[0]['rejected']

'```\n{\n    "problems": [\n        {\n            "area": "Enterprise",\n            "nature": "phishing",\n            "description": "Employees receive emails that appear to come from internal IT requesting credential updates. The messages contain malicious links that harvest login credentials when clicked.",\n            "risk_reduction": [\n                "Enable multi‑factor authentication (MFA) for all user accounts",\n                "Deploy email filtering to flag or quarantine suspicious senders",\n                "Run regular security awareness training and phishing simulations"\n            ]\n        },\n        {\n            "area": "Enterprise",\n            "nature": "insider_data_exfiltration",\n            "description": "A trusted employee copies sensitive customer data onto a personal USB drive and transfers it out of the corporate network.",\n            "risk_reduction": [\n                "Implement strict USB/device usage policies with endpoint DLP",\n        

In [96]:
## 3. DEFINE CUSTOM REWARD FUNCTIONS
# ======================================
# This is the core of our custom GRPO setup. We define three functions that
# will score the model's generated outputs. The total reward for any given
# completion is the sum of the scores from all these functions.

# Load a sentence transformer model once for efficient semantic comparison.
similarity_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')

def extract_json_from_string(text: str) -> str | None:
    """A helper function to find and extract a JSON list from a larger string."""
    match = re.search(r'(\[.*\])', text, re.DOTALL)
    return match.group(1) if match else None

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Gives a large, definitive reward (2.0) only if the generated JSON
    is an exact match to the ground-truth answer. This is a strong incentive
    to learn the precise correct output.
    """
    rewards = []
    for completion, ground_truth_str in zip(completions, answer):
        extracted_str = extract_json_from_string(completion)
        if extracted_str and extracted_str == ground_truth_str:
            rewards.append(2.0)
        else:
            rewards.append(0.0)
    return rewards

def valid_json_reward_func(completions, **kwargs) -> list[float]:
    """
    Gives a small partial reward (0.5) if the output contains any
    syntactically valid JSON. This encourages the model to learn the
    correct format, even if the content isn't perfect yet.
    """
    rewards = []
    for completion in completions:
        extracted_str = extract_json_from_string(completion)
        if extracted_str:
            try:
                json.loads(extracted_str)
                rewards.append(0.5)
                continue
            except json.JSONDecodeError:
                pass
        rewards.append(0.0)
    return rewards

def semantic_similarity_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    **IMPROVED**: Gives a nuanced reward based on how semantically similar the
    generated content is to the ground truth. It extracts only the text values
    from the JSON, ignoring syntax, for a cleaner and more accurate comparison.
    """
    def get_semantic_content(json_string: str) -> str:
        """Helper to parse JSON and extract text content for comparison."""
        try:
            data = json.loads(json_string)
            if isinstance(data, list):
                # Join all string values from all dicts in the list
                return " ".join(str(v) for item in data if isinstance(item, dict) for v in item.values())
        except (json.JSONDecodeError, TypeError):
            return ""
        return ""

    generated_contents = [get_semantic_content(extract_json_from_string(c) or "") for c in completions]
    real_contents = [get_semantic_content(a) for a in answer]

    # Filter out empty strings that resulted from parsing errors
    valid_indices = [i for i, g in enumerate(generated_contents) if g]
    if not valid_indices:
        return [0.0] * len(completions)

    valid_generated = [generated_contents[i] for i in valid_indices]
    valid_real = [real_contents[i] for i in valid_indices]

    # Encode the clean text and compute similarity
    generated_embeddings = similarity_model.encode(valid_generated, convert_to_tensor=True)
    real_embeddings = similarity_model.encode(valid_real, convert_to_tensor=True)
    cosine_scores = util.cos_sim(generated_embeddings, real_embeddings)

    final_rewards = [0.0] * len(completions)
    for i, score_tensor in zip(valid_indices, torch.diag(cosine_scores)):
        final_rewards[i] = score_tensor.item() # Assign similarity score (0.0 to 1.0)
    return final_rewards

print("✅ Custom reward functions defined and improved.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Custom reward functions defined and improved.


In [43]:


# In[6]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")



GPU = Tesla T4. Max memory = 14.741 GB.
9.084 GB of memory reserved.


In [99]:
## 4. CONFIGURE AND RUN THE GRPO TRAINER
# ==================================
# We initialize the GRPOTrainer, passing our list of custom reward functions.
# The configuration is set with appropriate hyperparameters for your task.

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[
        correctness_reward_func,
        valid_json_reward_func,
        semantic_similarity_reward_func,
    ],
    train_dataset=dataset,
    # **FIXED**: `sft_weight` is an argument for the GRPOTrainer, not the GRPOConfig.
    # It has been moved here from the args section below.
    sft_weight=0.3,  # very important less value if we want creative and varaiation in our data generated. sft weight encourage to mimic the real answer as much as possible
    args=GRPOConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=40,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        beta=0.1,
        output_dir="outputs_grpo_final",
        report_to="none",
    ),
)
print("✅ GRPOTrainer initialized. Starting training...")

trainer.train()
print("🎉 Training complete!")


Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 2 to the `num_generations` of 8
✅ GRPOTrainer initialized. Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 20 | Total steps = 40
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std,rewards / valid_json_reward_func / mean,rewards / valid_json_reward_func / std,rewards / semantic_similarity_reward_func / mean,rewards / semantic_similarity_reward_func / std
1,0.000000,0.015625,0.044194,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.015625,0.088388,0.000000,0.000000
2,-0.000000,0.018487,0.052288,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.015625,0.088388,0.002862,0.016188
3,0.000000,0.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000038,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000100,0.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000566,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000200,0.015625,0.044194,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.001639,0.000000,0.000000,0.015625,0.088388,0.000000,0.000000
6,0.000300,0.047941,0.104076,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.003213,0.000000,0.000000,0.046875,0.148072,0.001066,0.006033
7,0.000900,0.140304,0.174187,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.008637,0.000000,0.000000,0.125000,0.219971,0.015304,0.038701
8,0.001800,0.301795,0.297850,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.017981,0.000000,0.000000,0.265625,0.253504,0.036170,0.047985
9,0.002900,0.484937,0.192937,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.028675,0.000000,0.000000,0.406250,0.198279,0.078687,0.060916
10,0.004000,0.568473,0.119138,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.039900,0.000000,0.000000,0.468750,0.122967,0.099722,0.039625


KeyboardInterrupt: 

In [108]:
df = pd.read_csv("/content/problems.csv")
user_prompt = df['user'][0]
system_prompt = df['system'][0]

In [109]:

# ## Inference
# Let's run the newly trained GRPO model.

# In[9]:
# Inference with the GRPO model
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content":system_prompt},
    {"role": "user", "content": user_prompt},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256, use_cache = True)



```
{
    "problems": [
        {
            "area": "Enterprise",
            "nature": "phishing",
            "description": "Employees receive emails that appear to come from internal IT requesting credential updates. The messages contain malicious links that harvest login credentials when clicked.",
            "risk_reduction": [
                "Enable multi‑factor authentication (MFA) for all user accounts",
                "Deploy email filtering to flag or quarantine suspicious senders",
                "Run regular security awareness training and phishing simulations"
            ]
        },
        {
            "area": "Enterprise",
            "nature": "insider_data_exfiltration",
            "description": "A trusted employee copies sensitive customer data onto a personal USB drive and transfers it out of the corporate network.",
            "risk_reduction": [
                "Implement strict USB/device usage policies with endpoint DLP",
                "Monitor fil

In [ ]:

# ## Saving the Model
# The process for saving LoRA adapters is the same as with SFT.

# In[10]:
model.save_pretrained("lora_model_grpo")
tokenizer.save_pretrained("lora_model_grpo")

# To save to GGUF, you can use the same commands as before
# if False: model.save_pretrained_gguf("model_grpo", tokenizer, quantization_method = "q4_k_m")